In [1]:
!pip install docling langchain-docling langchain-openai langchain-community faiss-cpu pillow requests --q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
clarifai 10.11.1 requires click==8.1.7, but you have click 8.4.1 which is incompatible.
crewai 0.28.8 requires langchain<0.2.0,>=0.1.10, but you have langchain 0.2.17 which is incompatible.
crewai 0.28.8 requires openai<2.0.0,>=1.13.3, but you have openai 3.3.1 which is incompatible.
crewai-tools 0.1.6 requires langchain<0.2.0,>=0.1.4, but you have langchain 0.2.17 which is incompatible.
crewai-tools 0.1.6 requires openai<2.0.0,>=1.12.0, but you have openai 3.3.1 which is incompatible.
embedchain 0.1.113 requires langchain<0.2.0,>=0.1.4, but you have langchain 0.2.17 which is incompatible.
embedchain 0.1.113 requires langchain-openai<0.2.0,>=0.1.7, but you have langchain-openai 1.6.0 which is incompatible.
google-genai 2.9.0 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.11.10 which is incompatible.
ins

In [2]:
import base64
import io
from pathlib import Path
from PIL import Image

# Docling Imports
from docling.chunking import HybridChunker
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
#from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer

# LangChain & OpenAI Imports
import tiktoken
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from transformers import AutoTokenizer

c:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Ankur\AppData\Local\Temp\ipykernel_22080\1808415339.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
import utils

In [4]:
# Initialize OpenAI Components
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [5]:
LOCAL_PDF_PATH = "ICICI Bank Loan Portfolio Report.pdf"

In [6]:
def pil_to_base64(pil_img: Image.Image) -> str:
    buffered = io.BytesIO()
    pil_img.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")


# ---------------------------------------------------------------------------
# Enhanced Vision Prompting
# ---------------------------------------------------------------------------
def generate_figure_summary(b64_image: str, vision_llm: ChatOpenAI) -> str:
    prompt = (
        "You are an expert financial analyst examining a visual from an investor presentation.\n"
        "1. Map every pie-chart or bar-graph slice/color explicitly to its label and exact percentage value.\n"
        "2. Do not confuse adjacent legend items or misalign percentage labels.\n"
        "3. EXPLICITLY specify the denominator (e.g., state whether a figure is '% of Total Loan Portfolio' "
        "or '% of Retail Portfolio').\n"
        "4. Output all extracted values clearly in a bulleted key-value list."
    )
    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{b64_image}"},
            },
        ]
    )
    return vision_llm.invoke([message]).content

# **Process PDF for RAG**

1. **Setting flags for Docling**
By default, standard PDF parsers only extract plain text streams. Setting the flags forces Docling’s vision and layout models to detect bounding boxes around embedded figures, pie charts, and complex tables, cropping them out as discrete image objects.

2. **Layout Parsing via Document Converter**
Docling runs deep-learning vision models to map out the entire structural hierarchy of the document. It distinguishes section headers, main body paragraphs, native tables, and full-page images.

3. **Context Aware Chunking**
Standard recursive splitters break text strictly by character length, often chopping off mid-sentence or splitting a key heading away from its body text.HybridChunker advantage: It uses Docling's structural understanding to group sentences by document sections while enforcing strict maximum token constraints derived directly from your target embedding model.

4. **Raw Markdown Table Extraction**
Converting PDF tables directly into clean Markdown grid structures (| Header 1 | Header 2 |) allows vector search models to keep row-and-column relationships intact.

5. **Multimodal Vision Processing for Elements**
Critical metrics inside pitch decks live in visual charts (pie charts, bar graphs, process flows) that plain text parsers completely ignore.

    How it works:

    a. It iterates through doc items to find visual elements (picture, table).

    b. Extracts the cropped image slice.

    c. Sends the cropped image to gpt-4o with a prompt that extracts legend keys, values, and precise denominators into plain text.

    d. Appends both the text summary and the Base64 image into vector store memory.

In [7]:
# ---------------------------------------------------------------------------
# Document Processing Pipeline using Docling
# ---------------------------------------------------------------------------
def process_pdf_for_rag(file_path: str):
    pdf_path = Path(file_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"Local file not found at: {pdf_path.resolve()}")

    print("1. Configuring Docling Pipeline for layout analysis & figure cropping...")
    pipeline_options = PdfPipelineOptions()
    pipeline_options.generate_picture_images = True
    pipeline_options.generate_table_images = True

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    print(f"2. Parsing document: {pdf_path.name}")
    conversion_result = converter.convert(str(pdf_path))
    docling_doc = conversion_result.document

    documents = []

    # =========================================================================
    # A. STRUCTURED CHUNKING: Docling HybridChunker
    # =========================================================================
    print("3. Chunking document text using layout-aware HybridChunker...")

    # Uses OpenAI's exact native tokenizer instead of Hugging Face
    tokenizer = OpenAITokenizer(tokenizer=tiktoken.encoding_for_model("text-embedding-3-small"),
                                max_tokens=512)

    chunker = HybridChunker(
        tokenizer=tokenizer,
        max_tokens=512,
        merge_peers=True
    )

    chunk_iter = chunker.chunk(docling_doc)

    for idx, chunk in enumerate(chunk_iter):
        headings = " > ".join(chunk.meta.headings) if hasattr(chunk.meta, "headings") and chunk.meta.headings else "General"
        content_with_context = f"[Context: {headings}]\n{chunk.text}"

        documents.append(
            Document(
                page_content=content_with_context,
                metadata={
                    "type": "text",
                    "chunk_id": idx,
                    "headings": headings,
                    "source": str(pdf_path.name),
                },
            )
        )

    # =========================================================================
    # B. STRUCTURED TABLES: Raw Markdown Export
    # =========================================================================
    print("4. Processing structured markdown tables...")
    for table_idx, table in enumerate(docling_doc.tables):
        table_md = table.export_to_markdown()
        documents.append(
            Document(
                page_content=f"[Structured Financial Table #{table_idx + 1}]\n{table_md}",
                metadata={
                    "type": "table_text",
                    "table_id": table_idx + 1,
                    "source": str(pdf_path.name),
                },
            )
        )

    # =========================================================================
    # C. VISUAL FIGURES & CHARTS: Image Crops + GPT-4o Summarizer
    # =========================================================================
    print("5. Extracting and analyzing visual charts and diagrams...")
    figure_count = 0
    for element, level in docling_doc.iterate_items():
        if getattr(element, "label", None) in ["picture", "table"]:
            figure_count += 1
            img = element.get_image(docling_doc)

            if img:
                b64_img = pil_to_base64(img)
                vision_summary = generate_figure_summary(b64_img, llm)

                documents.append(
                    Document(
                        page_content=f"[Extracted Visual Component #{figure_count}]\n{vision_summary}",
                        metadata={
                            "type": "image",
                            "figure_id": figure_count,
                            "image_b64": b64_img,
                            "source": str(pdf_path.name),
                        },
                    )
                )

    print(f"   --> Processed {len(documents)} total chunks ({figure_count} visual components).")
    return documents

In [12]:
# ---------------------------------------------------------------------------
# Indexing & Multi-Modal Query Pipeline
# ---------------------------------------------------------------------------
documents = process_pdf_for_rag(LOCAL_PDF_PATH)

print("6. Creating FAISS Vector Database...")
vectorstore = FAISS.from_documents(documents, embeddings)

1. Configuring Docling Pipeline for layout analysis & figure cropping...
2. Parsing document: ICICI Bank Loan Portfolio Report.pdf



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "c:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.sta

AttributeError: _ARRAY_API not found

[INFO] 2026-08-23 11:19:42,261 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-23 11:19:42,263 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-23 11:19:42,305 [RapidOCR] download_file.py:60: File exists and is valid: C:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-23 11:19:42,308 [RapidOCR] main.py:50: Using C:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-23 11:19:43,479 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-23 11:19:43,482 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-23 11:19:43,491 [RapidOCR] download_file.py:60: File exists and is valid: C:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-23 11:19:43,494 [RapidOCR] main.py:50: Using C:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\rapidocr\model

3. Chunking document text using layout-aware HybridChunker...


Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.


4. Processing structured markdown tables...
5. Extracting and analyzing visual charts and diagrams...
   --> Processed 40 total chunks (20 visual components).
6. Creating FAISS Vector Database...


Handling Text Chunks (text / table_text): Appends plain markdown paragraphs and table strings directly to messages_content as standard text blocks.

Handling Image Chunks (image):

Appends the pre-extracted visual text summary (generated during ingestion by generate_figure_summary).

Injects the actual Base64-encoded image crop directly via an image_url block. This allows gpt-4o to cross-reference text with raw pixels.

In [9]:
def query_financial_rag(query: str, vectorstore: FAISS, vision_llm: ChatOpenAI):
    #print(f"\n--- Querying System: '{query}' ---")
    retrieved_docs = vectorstore.similarity_search(query, k=4)

    messages_content = [
        {
            "type": "text",
            "text": (
                f"User Question: {query}\n\n"
                "Synthesize a data-backed answer using the retrieved context from ICICI Bank's report. "
                "Verify whether percentages are '% of Total Loans' or '% of Retail Loans'."
            ),
        }
    ]

    for doc in retrieved_docs:
        doc_type = doc.metadata.get("type")

        if doc_type in ["text", "table_text"]:
            messages_content.append(
                {
                    "type": "text",
                    "text": f"\n--- Retrived Snippet ---\n{doc.page_content}",
                }
            )
        elif doc_type == "image":
            b64_img = doc.metadata.get("image_b64")
            messages_content.append(
                {
                    "type": "text",
                    "text": f"\n--- Retrieved Visual Context ---\n{doc.page_content}",
                }
            )
            messages_content.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{b64_img}"},
                }
            )

    response = vision_llm.invoke([HumanMessage(content=messages_content)])
    return response.content

In [10]:
def query_response(qry):
  prompt = f"Answer the following question directly: {qry}. Do not give any computation"
  answer = query_financial_rag(prompt, vectorstore, llm)
  return answer

In [11]:
query_response("What percentage of loan portfolio are mortgages")


NameError: name 'vectorstore' is not defined

In [ ]:
query_response("What percentage of loan portfolio are personal loans")


In [ ]:
query_response("For the corporate portfolio what % are unrated as on 30-Jun-2026")


In [ ]:
query_response("For the corporate portfolio what % are unrated as on 30-Jun-2025")
